# Reto: Contexto con imágenes

En el juego *Contexto* hay una palabra secreta y el jugador recibe, por cada intento, qué tan cerca está de ella. En este reto construirán la versión con imágenes: la computadora elige en secreto una imagen de una galería y el jugador la describe con texto hasta que su descripción ponga la imagen secreta en el **lugar 1** de la galería.

Esta libreta solo prepara el modelo y la galería. La lógica del juego, el puntaje y la interfaz los construye el equipo. Funciona en CPU; no es necesario activar una GPU.

In [ ]:
!pip install -q open_clip_torch gradio

import io, random, tarfile, urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)

## Modelo

Se usa OpenCLIP ViT-B/32, una versión de CLIP entrenada con datos públicos (LAION-400M). `clip.encode_image` y `clip.encode_text` producen embeddings de 512 dimensiones; `preprocesar` prepara una imagen y `tokenizar` prepara una lista de textos. CLIP se entrenó con descripciones en inglés.

In [ ]:
import open_clip

clip, _, preprocesar = open_clip.create_model_and_transforms(
    "ViT-B-32-quickgelu", pretrained="laion400m_e32")
tokenizar = open_clip.get_tokenizer("ViT-B-32-quickgelu")
clip = clip.to(dispositivo).eval()
print("CLIP listo.")

## Galería

La galería contiene 100 imágenes de Imagenette, un subconjunto de ImageNet: 10 imágenes de cada una de 10 categorías. `galeria[i]` es una imagen y `categorias[i]` es su categoría, que pueden usar, por ejemplo, como pista. Pueden sustituirla por una galería propia de al menos 50 imágenes.

In [ ]:
CATEGORIAS = {"n01440764": "tench", "n02102040": "English springer", "n02979186": "cassette player",
              "n03000684": "chain saw", "n03028079": "church", "n03394916": "French horn",
              "n03417042": "garbage truck", "n03425413": "gas pump", "n03445777": "golf ball",
              "n03888257": "parachute"}

def indexar_imagenette(destino="imagenette"):
    """Descarga Imagenette (160 px) y agrupa las imágenes de validación por carpeta."""
    url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
    raiz = Path(destino)
    marca = raiz / ".descarga_completa"              # evita usar una extracción interrumpida
    if not marca.exists():
        with urllib.request.urlopen(url) as r:
            tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz").extractall(raiz, filter="data")
        marca.touch()
    grupos = {}
    for ruta in raiz.rglob("*"):
        if ruta.suffix.lower() in (".jpeg", ".jpg", ".png") and any(x.startswith("val") for x in ruta.parts):
            grupos.setdefault(ruta.parent.name, []).append(ruta)
    return {carpeta: sorted(rutas) for carpeta, rutas in grupos.items()}

def cargar_galeria(por_carpeta=10, semilla=0):
    """Regresa las imágenes de la galería y la categoría de cada una."""
    grupos = indexar_imagenette()
    rng = random.Random(semilla)
    imagenes, categorias = [], []
    for carpeta, categoria in CATEGORIAS.items():
        rutas = grupos.get(carpeta, [])
        if len(rutas) < por_carpeta:
            raise RuntimeError(f"Solo se encontraron {len(rutas)} imágenes de la carpeta {carpeta}. "
                               f"Carpetas encontradas: {sorted(grupos)[:12]}")
        for ruta in rng.sample(rutas, por_carpeta):
            imagenes.append(Image.open(ruta).convert("RGB"))
            categorias.append(categoria)
    return imagenes, categorias

galeria, categorias = cargar_galeria()
print(len(galeria), "imágenes en la galería")

fig, ejes = plt.subplots(2, 8, figsize=(16, 5))
for eje, i in zip(ejes.flat, random.Random(1).sample(range(len(galeria)), 16)):
    eje.imshow(galeria[i]); eje.set_title(categorias[i], fontsize=9); eje.axis("off")
plt.tight_layout(); plt.show()

## Su juego

A partir de aquí, el código es del equipo. Los requisitos mínimos son:

1. Calcular una sola vez los embeddings de todas las imágenes de la galería.
2. Elegir al azar una imagen secreta al iniciar cada partida.
3. Por cada descripción del jugador, calcular en qué lugar queda la imagen secreta cuando la galería se ordena de la más parecida a la menos parecida a esa descripción.
4. Mostrar el historial de intentos con su lugar y declarar la victoria cuando la imagen secreta quede en el lugar 1.
5. Construir una interfaz con Gradio que otra persona pueda jugar desde el enlace público que genera Colab.

Antes de la demostración, cualquier integrante del equipo debe poder explicar cómo se calcula el lugar de la imagen secreta.

In [ ]:
# Código del equipo